In [ ]:
# Homework 3 Problem 2
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import CubicSpline

def dv_h(R):
    """ 
    Delta V magnitude for a standard hohmann transfer
    """
    vh = 2*np.pi*(np.sqrt(2*R/(1+R)) - 1 + np.sqrt(1/R) - np.sqrt(2/(R*(1+R))))
    return vh

def dv_bp(R):
    """ 
    Delta V magnitude for a bi parabolic transfer
    """
    vbp = 2*np.pi*(np.sqrt(2)-1)*(1+1/np.sqrt(R))
    return vbp

R = np.linspace(2,100,100)

ratio = [dv_bp(r)/dv_h(r) for r in R]

# Finding R*
from scipy.optimize import fsolve

# Define the function for which to find the root
def ratio_func(R):
    return dv_bp(R)/dv_h(R) - 1  

# Provide an initial guess
initial_guess = 3

# Find the root
root = fsolve(ratio_func, initial_guess)
R_star = root[0]
print(R_star)

fig = go.Figure()

# Add a line trace
fig.add_trace(go.Scatter(x=R, y=ratio, mode='lines',name='Ratio'))
fig.add_trace(go.Scatter(x=[R_star], y=[1],mode='markers',marker=dict(color='red',size=12, symbol='circle'),name='R*'))

# Update layout for title and axis labels
fig.update_layout(title='Ratio of Bi-Parabollic cost to Hohmann vs R',
                  xaxis_title='R (AU)',
                  yaxis_title='DV_bp/DV_h')

# Display the graph
fig.show()





In [ ]:
def dv_be(R,L):
    """ 
    Cost of a bi elliptic transfer from 1 AU to R with an outer radius of L
    """

    dv = np.sqrt(2*L/(1+L)) - 1 + np.sqrt(2*R/((R+L)*L)) - np.sqrt(2/(L+L**2)) + np.sqrt(2*L/(R**2+R*L)) - np.sqrt(1/R)
    return 2*np.pi*dv

R = [9,10,11,12,13,14,15]

fig = go.Figure()

ratios = []
guesses = [10,10,10,10,40,50,50]
mins = [1,1,1,1,1,0.99,0.99]
for i in range(len(R)):
    r = R[i]
    L = np.linspace(r,100,100-r) 
    ratio = [dv_be(r,l)/dv_h(r) for l in L]
    ratios.append(ratio)

    def ratio_func(l):
        return dv_be(r,l)/dv_h(r) - 1  

    # Provide an initial guess
    initial_guess = guesses[i]
    print(f"initial guess = {initial_guess}")
    min_val = mins[i]
    # Find the root
    if any(x<min_val for x in ratio):
        print('conditional')
        root = fsolve(ratio_func, initial_guess)
        R_star = root[0]
        print(f"For R = {r} and L* = {R_star}")
    else:
        R_star = r
        print(f"For R = {r} and L* = {R_star}")

    # Add a line trace
    fig.add_trace(go.Scatter(x=L, y=ratio, mode='lines',name=f'R = {r}'))
    fig.add_trace(go.Scatter(x=[R_star], y=[1],mode='markers',marker=dict(size=12, symbol='circle'),name=f'R = {r}'))

# Update layout for title and axis labels
fig.update_layout(title='Ratio of Bi-Elliptic cost to Hohmann vs L',
                  xaxis_title='L (AU)',
                  yaxis_title='DV_bpe/DV_h')

# Display the graph
fig.show()
    


In [ ]:
import plotly.graph_objects as go
import numpy as np

R = np.linspace(2,100,100)
L = np.linspace(2,100,100)

ratios = []
for i in range(len(R)):
    r = R[i]
    sl = []
    for j in range(len(L)):
        l = L[j]
        
        if l<=r:
            sl.append(dv_bp(r)/dv_be(r,l))
        else:
            sl.append(0)
    ratios.append(sl)

# Create the contour plot
fig = go.Figure(data=go.Contour(z=ratios,
        colorbar=dict(title='dv_bp/dv_be')))

fig.update_layout(title='Comparison of Maneuver Cost',
                  xaxis_title='R (AU)',
                  yaxis_title='L (AU)')


fig.show()

# Creating conditional plot
cond = []
for i in range(len(ratios)):
    cond.append([1 if ratio<1 else 0 for ratio in ratios[i]])


fig = go.Figure(data=[
    go.Contour(
        x=R,
        y=L,
        z=cond,  # Use the conditional Z data for coloring
        colorscale=[
            [0, 'red'],  # For values below the condition
            [0.5, 'blue'],
            [1, 'green']  # For values meeting the condition
        ],
        colorbar=dict(title='Conditional Value')
    )
])

fig.update_layout(title='Regions Where Bi-Parabollic is Cheaper',
                  xaxis_title='R (AU)',
                  yaxis_title='L (AU)')

fig.show()


    


In [ ]:
import numpy as np

def dv(r1,i,mu):
    """ 
    1 impulse plane change dv
    """
    return 2*np.sqrt(mu/r1)*np.sin(i/2)

def dv_t(rho,r1,i,mu):
    """ 
    3 impulse plane change dv
    """
    dv = np.sqrt(2*rho/(1+rho)) - 1 + np.sin(i/2)*np.sin(2/(rho*(1+rho)))
    return 2*np.sqrt(mu/r1)*dv

def R(rho,r1,i,mu):
    """ 
    Ratio of VT/V
    """
    return dv_t(rho,r1,i,mu)/dv(r1,i,mu)

mu = 4e5 # km^3/s^2
rho = [2,5,'infinity']
r1 = 600 + 6400
i = [60,45,30]
inc = [x*np.pi/180 for x in i]
dv_t_infty = 2*np.sqrt(mu/r1)*(np.sqrt(2)-1)


for i in inc:
    for r in rho:
        if r != 'infinity':
            dvt = dv_t(r,r1,i,mu)
            dv0 = dv(r1,i,mu)
            ratio = R(r,r1,i,mu)
            print(f"i = {round(i*180/np.pi)}, rho = {round(r,4)}: dv_1 = {round(dv0,4)}, dv_T = {round(dvt,4)},  ratio = {round(ratio,4)}")
        else:
            dvt = 2*np.sqrt(mu/r1)*(np.sqrt(2)-1)
            dv0 = dv(r1,i,mu)
            ratio = dvt/dv0
            print(f"i = {round(i*180/np.pi)}, rho = {r}: dv_1 = {round(dv0,4)}, dv_T = {round(dvt,4)},  ratio = {round(ratio,4)}")





In [ ]:
from scipy.optimize import fsolve
import plotly.graph_objects as go
import numpy as np
def rho_inf(i):
    """ 
    Cost of bi-elliptic at rho=infty
    """
    return (np.sqrt(2)-1)*1/np.sin(i/2)

id = np.linspace(5,90,100)
ratios = [rho_inf(x*np.pi/180) for x in id]

# Provide an initial guess
initial_guess = 49

ratio_func = lambda x: rho_inf(x) - 1
# Find the root
root = fsolve(ratio_func, initial_guess)
R_star = root[0]
print(R_star)

fig = go.Figure()

# Add a line trace
fig.add_trace(go.Scatter(x=id, y=ratios, mode='lines',name='Ratio'))
fig.add_trace(go.Scatter(x=[R_star], y=[1],mode='markers',marker=dict(color='red',size=12, symbol='circle'),name='R*'))

# Update layout for title and axis labels
fig.update_layout(title='Ratio of 3 to 1 burn Maneuvers for Inclination i (rho=infty)',
                  xaxis_title='i (deg)',
                  yaxis_title='DV_T/DV ')

# Display the graph
fig.show()


In [ ]:
from scipy.optimize import fsolve
import plotly.graph_objects as go
import numpy as np

def esc_ratio(ratio,mu,l,r):
    """ 
    Find ratio of DV_II/DV_I
    """
    dv1 = np.sqrt(mu / r) * (np.sqrt(ratio**2 + 2) - 1)
    term1 = 1 - np.sqrt(2 * l / (1 + l))
    term2 = np.sqrt(ratio**2 + 2 / l)
    term3 = np.sqrt(2 / (l * (1 + l)))
    dv2 =  np.sqrt(mu / r) * (term1 + term2 - term3)
    return dv2/dv1

vrat = np.linspace(0,20,50)
ls1 = np.linspace(0.1,1,8)
mu = 4e5
r = 6400+600
solution = []


fig = go.Figure()

for l in ls1:
    solution.append([esc_ratio(x,mu,l,r) for x in vrat])
    fig.add_trace(go.Scatter(x=vrat, y=solution[-1], mode='lines',name=f'l = {round(l,3)}'))

# Update layout for title and axis labels
fig.update_layout(title='Ratio of 2 Burn to 1 Burn Maneuver to Escape Velocity',
                  xaxis_title='V_infty/V_lc',
                  yaxis_title='DV_II/DV_I ')
fig.show()

ls2 = np.linspace(0.1,1,50)
fig = go.Figure()
solution = []
for v in vrat:
    solution.append([esc_ratio(v,mu,l,r) for l in ls2])
fig = go.Figure(data=go.Contour(z=solution,x=vrat,y=ls2,
        colorbar=dict(title='DV_II/DV_I')))

fig.update_layout(title='Comparison of Maneuver Cost',
                  xaxis_title='V_infty/V_lc(r)',
                  yaxis_title='l')


fig.show()

# Solving for the root numerically
# Provide an initial guess
initial_guess = 1
l=0.5
ratio_func = lambda x: esc_ratio(x,mu,l,r) - 1
# Find the root
root = fsolve(ratio_func, initial_guess)
R_star = root[0]
print(R_star)
    